# Whisper large-v3-turbo — Darija Code-Switch Fine-tuning

**Strategy:** Continue training `anaszil/whisper-large-v3-turbo-darija` (r=16, 24.88% WER baseline) on your 32k code-switched dataset.

**Requirements:**
- Colab Pro with High-RAM runtime (52GB) — needed for 32k sample preprocessing
- Your HF token with read access to `afyfbadreddine77/darija-asr-augmented`

**Steps:** Install → Load model → Preprocess dataset → Train → Push to Hub

In [ ]:
# ── Cell 1: Install (run once, then Runtime > Restart session) ────────────────
import os, re
import torch

v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post1'}.get(v, '0.0.34')

!pip install sentencepiece protobuf "datasets==3.6.0" huggingface_hub
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install librosa soundfile evaluate jiwer "datasets==3.6.0"
!pip install unsloth

In [ ]:
# ── Cell 2: Config — set your HF token here ───────────────────────────────────
HF_TOKEN      = "hf_YOUR_TOKEN_HERE"   # <-- paste your HF token
DATASET_NAME  = "afyfbadreddine77/darija-asr-augmented"
BASE_MODEL    = "openai/whisper-large-v3-turbo"
ADAPTER_MODEL = "anaszil/whisper-large-v3-turbo-darija"
HF_REPO       = "afyfbadreddine77/whisper-large-v3-turbo-darija"
LANGUAGE      = "arabic"
LANG_TOKEN    = "<|ar|>"

In [ ]:
# ── Cell 3: Load base model + existing Darija adapter ─────────────────────────
from unsloth import FastModel, is_bf16_supported
from transformers import WhisperForConditionalGeneration
from peft import PeftModel
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Load base model via Unsloth (2x faster training)
model, tokenizer = FastModel.from_pretrained(
    model_name   = BASE_MODEL,
    dtype        = None,
    load_in_4bit = False,
    auto_model   = WhisperForConditionalGeneration,
    whisper_language = LANGUAGE,
    whisper_task = "transcribe",
    token        = HF_TOKEN,
)

# Load existing Darija adapter on top
model = PeftModel.from_pretrained(model, ADAPTER_MODEL, is_trainable=True, token=HF_TOKEN)

model.generation_config.language           = LANG_TOKEN
model.generation_config.task               = "transcribe"
model.config.suppress_tokens               = []
model.generation_config.forced_decoder_ids = None

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ── Cell 4: Preprocess dataset into memory ────────────────────────────────────
# NOTE: Requires High-RAM runtime (52GB) in Colab Pro.
# If you get OOM here, go to Runtime > Change runtime type > High-RAM
import numpy as np
import tqdm
from datasets import load_dataset, Audio

raw = load_dataset(DATASET_NAME, token=HF_TOKEN)
raw = raw.cast_column("audio", Audio(sampling_rate=16000))

def preprocess(example):
    features = tokenizer.feature_extractor(
        example["audio"]["array"],
        sampling_rate=example["audio"]["sampling_rate"],
    )
    label_ids = tokenizer.tokenizer(example["text"].strip()).input_ids
    return {"input_features": features.input_features[0], "labels": label_ids}

print("Preprocessing train (32k samples, ~15 min)...")
train_dataset = [preprocess(ex) for ex in tqdm.tqdm(raw["train"])]
print("Preprocessing validation...")
val_dataset   = [preprocess(ex) for ex in tqdm.tqdm(raw["validation"])]

print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}")

In [ ]:
# ── Cell 5: Data collator + metrics ───────────────────────────────────────────
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

metric = evaluate.load("wer")

def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.tokenizer.pad_token_id
    # predict_with_generate=True: predictions are already token IDs, not logits
    pred_ids  = pred.predictions if isinstance(pred.predictions, np.ndarray) else pred.predictions[0]
    pred_str  = tokenizer.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=tokenizer)

In [ ]:
# ── Cell 6: Train ─────────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model         = model,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    tokenizer     = tokenizer.feature_extractor,
    args = Seq2SeqTrainingArguments(
        output_dir                  = "outputs",
        per_device_train_batch_size = 16, # reduced to leave headroom for eval generation
        per_device_eval_batch_size  = 8,  # generation needs ~4x more memory than training
        gradient_accumulation_steps = 2,   # effective batch = 32
        num_train_epochs            = 8,
        warmup_steps                = 200,
        learning_rate               = 1e-4,
        lr_scheduler_type           = "cosine",
        fp16                        = not is_bf16_supported(),
        bf16                        = is_bf16_supported(),
        optim                       = "adamw_torch",  # no need for 8-bit on 102GB
        weight_decay                = 0.01,
        logging_steps               = 25,
        eval_strategy               = "steps",
        eval_steps                  = 500,
        save_strategy               = "steps",
        save_steps                  = 500,
        save_total_limit            = 2,
        predict_with_generate       = True,   # required for correct WER computation
        generation_max_length       = 448,
        load_best_model_at_end      = True,
        metric_for_best_model       = "wer",
        greater_is_better           = False,
        remove_unused_columns       = False,
        label_names                 = ["labels"],
        seed                        = 3407,
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# ── Cell 7: Push LoRA adapters to HF Hub ──────────────────────────────────────
model.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
print(f"Done! https://huggingface.co/{HF_REPO}")

In [ ]:
# ── Cell 8: Evaluation on test split (overall + per lang_mix) ─────────────────
from unsloth import FastModel
import torch, numpy as np
from datasets import load_dataset, Audio
from evaluate import load as load_metric

wer_metric = load_metric("wer")
model.eval()

raw_test = load_dataset(DATASET_NAME, split="test", token=HF_TOKEN)
raw_test = raw_test.cast_column("audio", Audio(sampling_rate=16000))

def transcribe(example):
    feats = tokenizer.feature_extractor(
        example["audio"]["array"], sampling_rate=16000, return_tensors="pt"
    ).input_features.to(model.device)
    with torch.no_grad():
        ids = model.generate(
            feats,
            language="arabic", task="transcribe",
            max_new_tokens=444,
        )
    return tokenizer.tokenizer.decode(ids[0], skip_special_tokens=True)

refs, hyps, lang_mixes = [], [], []
for ex in tqdm.tqdm(raw_test):
    hyps.append(transcribe(ex))
    refs.append(ex["text"].strip())
    lang_mixes.append(ex["lang_mix"])

# Overall WER
overall_wer = 100 * wer_metric.compute(predictions=hyps, references=refs)
print(f"Overall WER: {overall_wer:.2f}%")

# WER per lang_mix category
for lang in sorted(set(lang_mixes)):
    idx = [i for i, l in enumerate(lang_mixes) if l == lang]
    w = 100 * wer_metric.compute(
        predictions=[hyps[i] for i in idx],
        references=[refs[i] for i in idx]
    )
    print(f"  {lang:20s} ({len(idx):3d} samples): WER = {w:.2f}%")